## 面试问题

单步怎么做成纯函数以支持单测与回放？

## 回答主线

纯函数步 `step(state, observation) -> (new_state, action)`：相同输入必得相同输出，内部不产生副作用、不读全局、不用时间/随机。所有不确定性作为 observation 注入。本 Notebook 用「按注入的挂起时长决定是否催单」的纯 step，回放两次得一致结果；再用一个内部读变化计数器的不纯 step，展示回放不一致、不可复现。

## 真实案例

催单决策：挂起超过 24 小时就催单。纯版本把 `pending_hours`/`now` 作为 observation 注入，回放确定；不纯版本内部读一个每次调用都变的计数器。数据为教学序列，不代表真实模型。

In [1]:
observations = [  # 定义一段固定的 observation 序列用于回放。
    {"pending_hours": 10, "now": 9},  # 第一步：挂起 10 小时。
    {"pending_hours": 30, "now": 14},  # 第二步：挂起 30 小时。
    {"pending_hours": 50, "now": 20},  # 第三步：挂起 50 小时。
]  # 结束 observation 序列定义。

print("observation 步数:", len(observations))  # 展示回放序列长度。
for ob in observations:  # 逐条打印注入的观察。
    print("  观察:", ob)  # 展示每步注入的时长与时间。

observation 步数: 3
  观察: {'pending_hours': 10, 'now': 9}
  观察: {'pending_hours': 30, 'now': 14}
  观察: {'pending_hours': 50, 'now': 20}


## 基线（Baseline）

纯 step 基线：决策只依赖注入的 observation，不读任何外部可变状态，因此对同一序列回放确定一致。

In [2]:
def pure_step(state, observation):  # 纯函数步：只依赖注入的 state 与 observation。
    should_remind = observation["pending_hours"] > 24  # 是否催单只由注入的观察决定。
    action = "remind" if should_remind else "wait"  # 依据阈值产出确定动作。
    new_state = {"reminders": state["reminders"] + (1 if should_remind else 0)}  # 不可变更新催单计数。
    return new_state, action  # 返回新状态与动作。

def replay(step_fn, observations):  # 用固定 observation 序列回放一条循环。
    state = {"reminders": 0}  # 初始化状态。
    actions = []  # 记录逐步动作。
    for ob in observations:  # 按序注入每个观察。
        state, action = step_fn(state, ob)  # 执行一步。
        actions.append(action)  # 记录动作。
    return state, actions  # 返回终态与动作序列。

In [3]:
run1_state, run1_actions = replay(pure_step, observations)  # 第一次回放纯 step。
run2_state, run2_actions = replay(pure_step, observations)  # 第二次回放纯 step。
print("纯 step 第一次动作:", run1_actions)  # 展示第一次回放结果。
print("纯 step 第二次动作:", run2_actions)  # 展示第二次回放结果。
print("两次是否一致:", run1_actions == run2_actions)  # 展示纯函数回放确定一致。

纯 step 第一次动作: ['wait', 'remind', 'remind']
纯 step 第二次动作: ['wait', 'remind', 'remind']
两次是否一致: True


## 失败案例与修正

不纯 step 在函数内部读取并修改一个外部计数器，决策依赖不可注入的内部状态。对同一 observation 序列回放两次，因计数器持续漂移，动作序列不同——bug 无法复现。修正就是把一切不确定性注入 observation，回到纯 step。

In [4]:
impure_counter = {"calls": 0}  # 定义一个会在步内变化的外部计数器。

def impure_step(state, observation):  # 不纯步：内部读取并修改外部计数器。
    impure_counter["calls"] += 1  # 每次调用都改变外部状态。
    should_remind = impure_counter["calls"] % 2 == 0  # 决策依赖不可注入的内部计数。
    action = "remind" if should_remind else "wait"  # 依内部计数产出动作。
    new_state = {"reminders": state["reminders"] + (1 if should_remind else 0)}  # 更新催单计数。
    return new_state, action  # 返回新状态与动作。

impure_run1_state, impure_run1_actions = replay(impure_step, observations)  # 第一次回放不纯 step。
impure_run2_state, impure_run2_actions = replay(impure_step, observations)  # 第二次回放不纯 step。
print("不纯 step 第一次动作:", impure_run1_actions)  # 展示第一次回放结果。
print("不纯 step 第二次动作:", impure_run2_actions)  # 展示第二次回放因内部状态漂移。
print("两次是否一致:", impure_run1_actions == impure_run2_actions)  # 展示不纯函数回放不可复现。

不纯 step 第一次动作: ['wait', 'remind', 'wait']
不纯 step 第二次动作: ['remind', 'wait', 'remind']
两次是否一致: False


## 结果解读

纯 step 两次回放动作都是 `[wait, remind, remind]`，完全一致、可复现；不纯 step 因内部计数器跨回放漂移，两次动作不同。结论：把模型输出、工具结果、时间、随机都注入 observation，才能让单步可单测、整条循环可确定性重放（题 27）。

In [5]:
print("纯 step 一致性:", run1_actions == run2_actions)  # 纯步回放一致。
print("不纯 step 一致性:", impure_run1_actions == impure_run2_actions)  # 不纯步回放不一致。
print("纯步动作:", run1_actions)  # 展示纯步动作。
print("不纯步两次:", impure_run1_actions, "vs", impure_run2_actions)  # 展示不纯步两次差异。

纯 step 一致性: True
不纯 step 一致性: False
纯步动作: ['wait', 'remind', 'remind']
不纯步两次: ['wait', 'remind', 'wait'] vs ['remind', 'wait', 'remind']


In [6]:
assert run1_actions == run2_actions  # 纯 step 两次回放必须一致。
assert run1_actions == ["wait", "remind", "remind"]  # 纯 step 动作由注入观察确定。
assert run1_state["reminders"] == 2  # 纯 step 终态可复现。
assert impure_run1_actions != impure_run2_actions  # 不纯 step 两次回放不一致。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
